# TFIM ground states: SVM balanced accuracies

Computes the balanced accuracies of Tables II and III (exact and finite-shot IPR features) and writes them to `figures/tables/tfim_svm.json`, which `figures/make_figures.py` reads for Fig. 3(c, d).

Requires `data/processed/tfim_1d/` from `process_data.ipynb`.

In [ ]:
from ipr_kernel.svm import run_iprkernel_svm_experiment, run_noisy_iprkernel_svm_ns_sweep
from ipr_kernel.tables import TABLES_DIR, svm_rows, write_table

In [ ]:
# Settings of the manuscript.
L_VALUES = list(range(10, 101, 10))
SHOTS = [200, 500, 1000, 2000, 5000, 10000]
N_SPLITS = 100          # exact features
N_SPLITS_NOISY = 10     # per noise realization
NOISE_SEEDS = list(range(10))
COMMON = dict(dataset="tfim1d", basis=("IPR X", "IPR Z"), phase_boundary=1.0, test_frac=0.10,
              local_delta=0.25, rng_seed=12345, use_normalized_kernel=True)

## Exact IPR features (Table II)

In [ ]:
_, _, exact = run_iprkernel_svm_experiment(L_values=L_VALUES, n_splits=N_SPLITS, make_plot=False,
                                           print_report=True, **COMMON)

## Finite-shot IPR features (Table III)

Gaussian noise with standard deviation $1/\sqrt{N_s}$, as in the finite-shot noise model of the appendix.

In [ ]:
_, _, noisy, _ = run_noisy_iprkernel_svm_ns_sweep(N_s_values=SHOTS, L_values=L_VALUES, noise_seeds=NOISE_SEEDS,
                                                n_splits=N_SPLITS_NOISY, make_plot=False, **COMMON)
noisy

## Write the table

In [ ]:
tables = {"exact": svm_rows(exact["L"], exact)}
for n_s, rows in noisy.groupby("N_s"):
    tables[str(n_s)] = [[int(r.L), *(round(float(v), 4) for v in (r.global_mean, r.global_std, r.local_mean, r.local_std))]
                        for r in rows.sort_values("L").itertuples()]
write_table(TABLES_DIR / "tfim_svm.json", tables,
            f"Balanced accuracies of tab:svm_tfim and tab:svm_tfim_noisy ({N_SPLITS} splits exact, {N_SPLITS_NOISY} splits x "
            f"{len(NOISE_SEEDS)} noise seeds finite-shot; local delta = 0.25). Columns: L, global mean, global std, "
            "local mean, local std.")